In [1]:
import pandas as pd
import numpy as np
from sqlalchemy.engine import URL, create_engine, Inspector

# Use your own server connection here
cnxn_str = ("Driver={SQL Server Native Client 11.0};"
            "Server=DAIYAAN;"
            "Database=AIFMRM_ERS;"
            "Trusted_Connection=yes;")

cnxn_url = URL.create("mssql+pyodbc", query={"odbc_connect": cnxn_str})

engine = create_engine(cnxn_url)



# initialise connection via context manager           
with engine.connect() as cnxn:
        tables_df = pd.read_sql('SELECT [name] AS [table_name] FROM sys.tables', cnxn)
        table_name_list = tables_df.table_name
        select_template = 'SELECT * FROM {table_name}'
        # Dictionary of table names and their respective SQL queries
        frames_dict = {}
        for tname in table_name_list:
                query = select_template.format(table_name = tname)
                frames_dict[tname] = pd.read_sql(query, cnxn)


"""
Keys in frames_dict Dictionary:
tbl_Industry_Classification_Benchmark
tbl_FTSEJSE_Index_Series 
tbl_EOD_Equity_Data
tbl_EOD_Interest_Rate_Data
tbl_Beta_Output 
tbl_BA_Beta_Output
tbl_Index_Constituents 
"""

'\nKeys in frames_dict Dictionary:\ntbl_Industry_Classification_Benchmark\ntbl_FTSEJSE_Index_Series \ntbl_EOD_Equity_Data\ntbl_EOD_Interest_Rate_Data\ntbl_Beta_Output \ntbl_BA_Beta_Output\ntbl_Index_Constituents \n'

## Function 1



In [4]:
def Function1(rDate,IndexCode):
    #Store tbl_Index_Constituents from frames_dict
    tbl_Index_Constituents = frames_dict["tbl_Index_Constituents"]

    #rDate will be supplied by the user: consisting of year and Quarter 
    rDate = rDate
    rDate = pd.to_datetime(rDate, format = "%Y-%m-%d")
    rDate_Quarter = rDate.quarter
    rDate_Year = rDate.year

    #search tbl_Index_Constituents Date column and find Quarter and Year for each date in column
    Dates_Col = tbl_Index_Constituents["Date"]
    Dates_Col = pd.arrays.DatetimeArray(Dates_Col)
    Dates_Col_Quarter = Dates_Col.quarter
    Dates_Col_Year = Dates_Col.year

    #Filter tbl_Index_Constituents using supplied quarter and year data from rData
    tbl_Index_Constituents_Date = tbl_Index_Constituents.loc[(Dates_Col_Quarter == rDate_Quarter) & (Dates_Col_Year == rDate_Year),]


    #IndexCode is provided by user: "ALSI", "FLED", "LRGC", "MIDC", "SMLC", "TOPI", "RESI", "FINI", "INDI", "PCAP", "SAPY" or "ALTI"
    IndexCode = IndexCode #provided as input by the user. "ALSI" for testing purposes

    #function to identify The index column that must be searched
    def Index_Col_Identifier(argument):
        match argument:
            case "FLED":
                return "ALSI New"
            case  "LRGG"|"MIDC"|"SMILC":
                return "Index New"
            case default:
                return argument+" New"
    
    IndexCode_Col = Index_Col_Identifier(IndexCode) #Obtain column name to search relevant rows

    #Filter tbl_Index_Constituents_Date using supplied IndexCode in the column identified from Index_Col_Identifier function
    tbl_Index_Constituents_final = tbl_Index_Constituents_Date[tbl_Index_Constituents_Date[IndexCode_Col] == IndexCode]


    Alpha = pd.DataFrame(tbl_Index_Constituents_final.loc[:,"Alpha"])
    Gross_Market_Capitalisation = np.array(tbl_Index_Constituents_final.loc[:,"Gross Market Capitalisation"])
    Weigths = pd.DataFrame(Gross_Market_Capitalisation/np.sum(Gross_Market_Capitalisation))
    Results = pd.concat([Alpha.reset_index(drop=True), Weigths.reset_index(drop=True)],axis=1)
    Results.columns = ['Alpha','Weights']
    return Results


rDate = '2019-6-05'
IndexCode = "ALSI"

Output = Function1(rDate,IndexCode)
print(Output)

    Alpha   Weights
0     ABG  0.010577
1     ACL  0.000225
2     ACT  0.000215
3     ADH  0.000595
4     AEL  0.000742
..    ...       ...
159   VKE  0.001414
160   VOD  0.016521
161   WBO  0.000512
162   WHL  0.003596
163   ZED  0.000486

[164 rows x 2 columns]
